<a href="https://colab.research.google.com/github/kkl5524/oasis-plus/blob/main/synthea_data_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# java
!apt-get update
!apt-get install -y openjdk-11-jdk-headless

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

!java -version

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,153 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,498 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-secu

In [2]:
# libraries and frameworks (not java)

import json
import pandas as pd
from datetime import datetime
import hashlib

In [3]:
# actual import of synthea

!git clone https://github.com/synthetichealth/synthea.git

Cloning into 'synthea'...
remote: Enumerating objects: 73471, done.
remote: Counting objects: 100% (1109/1109), done.
remote: Compressing objects: 100% (249/249), done.
remote: Total 73471 (delta 985), reused 864 (delta 859), pack-reused 72362 (from 3)
Receiving objects: 100% (73471/73471), 760.28 MiB | 27.41 MiB/s, done.
Resolving deltas: 100% (43769/43769), done.


In [4]:
# configurations for synthea (mimic-iii settings)

config_additions = """

exporter.mimic.export = true
exporter.mimic.folder = output/mimic

exporter.csv.export = true
exporter.baseDirectory = ./output/
exporter.use_uuid_filenames = false
exporter.subfolders_by_id_substring = false

exporter.years_of_history = 10

generate.demographics.socioeconomic.income.poverty = 18
generate.demographics.socioeconomic.education.less_than_hs.percentage = 10

generate.only_inpatient_encounters = true
generate.icustay.prob = 1.0
generate.icustay.mean_los_days = 3.5
generate.icustay.sd_los_days = 1.2

generate.population.minimum_age = 18

"""

with open('synthea/src/main/resources/synthea.properties', 'a') as f:
    f.write(config_additions)

print("Configuration updated")

Configuration updated


In [5]:
MODULE_DIR = "synthea/src/main/resources/modules/encounter"

In [6]:
# helper functions
def modify_json(path, callback):
    """Load → modify → save JSON"""
    with open(path, "r") as f:
        data = json.load(f)

    modified = callback(data)

    with open(path, "w") as f:
        json.dump(modified, f, indent=2)

def modify_icu_module(data):
    for state in data["states"].values():
        # longer ICU stays (MIMIC-style)
        if "distribution" in state and "mean" in state["distribution"]:
            state["distribution"]["mean"] = max(state["distribution"]["mean"], 3.5)
            state["distribution"]["minimum"] = 1.0

        # add higher chance of ventilation
        if "direct_transition" in state and state["direct_transition"] == "Ventilation":
            state["transition_probability"] = 0.35  # increase vent probability to MIMIC-like levels

    return data

def modify_hospital_module(data):
    for state in data.get("states", []):
        if isinstance(state, dict) and state.get("name") == "ICU":
            state["stay_distribution"] = {
                "type": "lognormal",
                "mean": 3.0,
                "stddev": 1.0
            }
    return data

def modify_vitals_module(data):
    for state in data["states"].values():
        if state.get("type") == "VitalSign":
            state["frequency"] = {"quantity": 15, "unit": "minutes"}  # MIMIC-style frequency
    return data

In [7]:
hospital_file = os.path.join(MODULE_DIR, "hospital_basic_labs.json")
vitals_file = os.path.join(MODULE_DIR, "vitals.json")

modify_json(hospital_file, modify_hospital_module)
modify_json(vitals_file, modify_vitals_module)

In [ ]:
!cd synthea && ./gradlew clean --no-daemon
!cd synthea && ./gradlew build -x test --no-daemon

.............10%.............20%.............30%.............40%.............50%.............60%.............70%.............80%.............90%..............100%

Welcome to Gradle 8.14.3!

Here are the highlights of this release:
 - Java 24 support
 - GraalVM Native Image toolchain selection
 - Enhancements to test reporting
 - Build Authoring improvements

For more details see https://docs.gradle.org/8.14.3/release-notes.html

To honour the JVM settings for this build a single-use Daemon process will be forked. For more on this, please refer to https://docs.gradle.org/8.14.3/userguide/gradle_daemon.html#sec:disabling_the_daemon in the Gradle documentation.


> Starting Daemon> IDLEDaemon will be stopped at the end of the build 

> IDLE<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------

In [ ]:
%cd synthea

In [ ]:
# Generate 5000 patients
!./run_synthea -s 123 -p 5000 Massachusetts

print("\nPatient generation complete")

Streaming output truncated to the last 5000 lines.
<==========---> 80% EXECUTING [1h 23m 38s]> :run<==========---> 80% EXECUTING [1h 23m 39s]7847 -- Miles206 Kovacek682 (33 y/o M) Natick, Massachusetts  (46403)

<==========---> 80% EXECUTING [1h 23m 39s]> :run7844 -- Han309 Yadira684 Reinger292 (63 y/o F) Saugus, Massachusetts DECEASED (87162)

<==========---> 80% EXECUTING [1h 23m 39s]> :run7848 -- Edmond919 Kling921 (7 y/o M) Brockton, Massachusetts  (10353)

<==========---> 80% EXECUTING [1h 23m 39s]> :run<==========---> 80% EXECUTING [1h 23m 40s]<==========---> 80% EXECUTING [1h 23m 41s]7849 -- Isela504 Selene142 Mertz280 (74 y/o F) Tyngsborough, Massachusetts  (102667)

<==========---> 80% EXECUTING [1h 23m 41s]> :run7850 -- Ruben688 Koss676 (15 y/o M) Holliston, Massachusetts  (20903)

<==========---> 80% EXECUTING [1h 23m 41s]> :run7851 -- Laurette750 Stamm704 (2 y/o F) Westminster, Massachusetts  (3584)
7844 -- Armanda758 Jast432 (89 y/o F) Saugus, Massachusetts  (181315)


<==

In [ ]:
# convert to mimic-iii format

input_folder = 'output/csv'
output_folder = 'output/mimic'

os.makedirs(output_folder, exist_ok=True)

In [ ]:
patients = pd.read_csv(os.path.join(input_folder, "patients.csv"))
patients.columns = patients.columns.str.lower()

encounters = pd.read_csv(os.path.join(input_folder, "encounters.csv"))
encounters.columns = encounters.columns.str.lower()

vitals = pd.read_csv(os.path.join(input_folder, "observations.csv"))
vitals.columns = vitals.columns.str.lower()

procedures = pd.read_csv(os.path.join(input_folder, "procedures.csv"))
procedures.columns = procedures.columns.str.lower()

In [ ]:
# PATIENTS.csv table

patients['birthdate'] = pd.to_datetime(patients['birthdate'])
patients['age'] = (datetime.now() - patients['birthdate']).dt.days // 365
mimic_patients = pd.DataFrame({
    'subject_id': patients['id'],
    'gender': patients['gender'],
    'anchor_age': patients['age'],
    'anchor_year': pd.to_datetime(patients['birthdate']).dt.year,
    'anchor_month': pd.to_datetime(patients['birthdate']).dt.month,
    'dod': patients['deathdate'],
})

mimic_patients.to_csv(os.path.join(output_folder, "PATIENTS.csv"), index=False)


In [ ]:
# ADMISSIONS.csv table

encounters = encounters[encounters['patient'].isin(patients['id'])]
mimic_adm = pd.DataFrame({
    'hadm_id': encounters['id'],
    'subject_id': encounters['patient'],
    'admittime': encounters['start'],
    'dischtime': encounters['stop'],
    'admission_type': encounters['encounterclass'],
    'diagnosis': encounters.get('reasondescription', None)
})

patients['death_date'] = pd.to_datetime(patients['deathdate'])
encounters['admittime'] = pd.to_datetime(encounters['start'])
encounters['dischtime'] = pd.to_datetime(encounters['stop'])

enc = encounters.merge(
    patients[['id', 'deathdate']],
    left_on='id', right_on='id', how='left'
)

enc['hospital_expire_flag'] = enc.apply(
    lambda row: 1 if pd.notna(row['deathdate']) and row['admittime'] <= row['deathdate'] <= row['dischtime'] else 0,
    axis=1
)

mimic_adm.to_csv(os.path.join(output_folder, "ADMISSIONS.csv"), index=False)


In [ ]:
# ICUSTAYS.csv table

icu_enc = encounters[encounters['encounterclass'] == 'inpatient'].copy()

icu_descriptions = [
    'Patient transfer to intensive care unit (procedure)',
    'Admission to intensive care unit (procedure)'
]

icu_enc = icu_enc[icu_enc['description'].isin(icu_descriptions)].copy()

mimic_icu = pd.DataFrame({
    'subject_id': icu_enc['patient'],
    'hadm_id': icu_enc['id'],
    'stay_id': range(1, len(icu_enc)+1),
    'intime': icu_enc['start'],
    'outtime': icu_enc['stop'],
    'los': (pd.to_datetime(icu_enc['stop']) - pd.to_datetime(icu_enc['start'])).dt.total_seconds()/86400
})
mimic_icu.to_csv(os.path.join(output_folder, "ICUSTAYS.csv"), index=False)


In [ ]:
# CHARTEVENTS.csv table

vitals = vitals[vitals['patient'].isin(patients['id'])]  # adult filter
mimic_chart = pd.DataFrame({
    'subject_id': vitals['patient'],
    'hadm_id': vitals.get('encounter', None),
    'icustay_id': None,
    'itemid': vitals['code'],
    'charttime': vitals['date'],
    'value': vitals['value'],
    'valuenum': pd.to_numeric(vitals['value'], errors='coerce'),
    'valueuom': vitals.get('units', None),
    'warning': None,
    'error': None
})
mimic_chart.to_csv(os.path.join(output_folder, "CHARTEVENTS.csv"), index=False)


In [ ]:
# LABEVENTS.csv table

labs = vitals[vitals['category'] == 'laboratory'].copy()
mimic_lab = pd.DataFrame({
    'subject_id': labs['patient'],
    'hadm_id': labs.get('encounter', None),
    'itemid': labs['code'],
    'charttime': labs['date'],
    'value': labs['value'],
    'valuenum': pd.to_numeric(labs['value'], errors='coerce'),
    'valueuom': labs.get('units', None),
    'flag': None
})
mimic_lab.to_csv(os.path.join(output_folder, "LABEVENTS.csv"), index=False)


In [ ]:
# PROCEDUREEVENTS_MV.csv table

procedures = procedures[procedures['patient'].isin(patients['id'])]
mimic_proc = pd.DataFrame({
    'subject_id': procedures['patient'],
    'hadm_id': procedures.get('encounter', None),
    'starttime': procedures['start'],
    'endtime': procedures['stop'],
    'itemid': procedures['code'],
    'value': procedures.get('description', None),
    'valueuom': None,
})
mimic_proc.to_csv(os.path.join(output_folder, "PROCEDUREEVENTS_MV.csv"), index=False)


In [ ]:
def map_category(code):
    label = str(code).lower()  # fallback to string search
    if any(x in label for x in ['vital-signs']):
        return 'Vital Signs'
    if any(x in label for x in ['laboratory']):
        return 'Laboratory'
    return "Other"

def map_fluid(code):
  label = str(code).lower()
  if any(x in label for x in ['blood', 'plasma']):
    return 'Blood'
  elif any(x in label for x in ['urine']):
    return 'Urine'
  elif any(x in label for x in ['csf', 'cerebrospinal']):
    return 'CSF'
  else:
    return None

def make_itemid(code):
    return int(hashlib.md5(code.encode()).hexdigest(), 16) % 10**6


In [ ]:
items = vitals[['category', 'code', 'description', 'units']].drop_duplicates()

items['itemid'] = items['code'].apply(make_itemid)
items['label'] = items['description']
items['category'] = items['category'].apply(map_category)
items['fluid'] = items['description'].apply(map_fluid)
items['unitname'] = items['units']
items['param_type'] = "Numeric"

mimic_items = items[['itemid','label','fluid','category','unitname','param_type']]
mimic_items.to_csv("D_ITEMS.csv", index=False)


In [ ]:
import zipfile

zip_path = "synthea_mimic_output.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk("synthea/output/mimic"):
        for file in files:
            full_path = os.path.join(root, file)
            zipf.write(full_path)


In [ ]:
from google.colab import files
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>